## 🎶 Feast 모델에서 예측 요청하기  

이제 우리는 기능을 가지고 있으므로, 배포된 모델을 사용하여 **예측을 만들** 차례입니다. 우리는 노래 데이터를 추론 엔드포인트에 보내고 반환으로 예측을 받습니다. 이 프로세스는 우리의 모델이 우리가 정의한 기능을 어떻게 해석하는지 이해하는 데 도움이 됩니다.

## 📥 의존성(Dependencies) 설정하기  

예측을 하기 전에, 우리는 데이터, 타임스탬프 및 API 요청을 처리하기 위한 **필요한 라이브러리를 가져와야** 합니다.

In [ ]:
import pandas as pd
from datetime import datetime
import yaml
import requests

## 🎯 추론 엔드포인트(Inference Endpoint) 정의하기  

배포된 모델과 상호작용하기 위해, 우리는 **모델 이름**(예: `"jukebox"`)과 **추론 API 엔드포인트**를 요청 전송 용도로 정의합니다.

In [ ]:
deployed_model_name = "jukebox"
infer_endpoint = "<paste-the-link-here>"
infer_url = f"{infer_endpoint}/v2/models/{deployed_model_name}/infer"

## 🎵 예측용 노래 선택하기  

우리의 모델을 테스트하려면, 먼저 우리의 테스트 케이스로 사용할 **특정 노래**를 선택해야 합니다. 우리는 다양한 노래 기능을 포함하는 **전처리된 데이터셋**을 로드함으로써 시작합니다. 

이 데이터셋에서, 우리는 특정 노래—예를 들어 `"Not Like Us"`—를 필터링하여 테스트 사례로 사용합니다. 선택한 노래를 얻으면, 우리는 우리의 모델 입력을 위한 고유 식별자로 작동하는 **Spotify ID**를 추출합니다. 

이것은 우리가 올바른 데이터를 우리의 추론 시스템에 보낼 수 있게 하고 우리가 인식하는 노래에 대해 의미 있는 예측을 얻을 수 있게 합니다.

In [ ]:
song_properties = pd.read_parquet('../99-data_prep/song_properties.parquet')
favorite_song = song_properties.loc[song_properties["name"]=="Not Like Us"]
favorite_song

## 🚀 예측 요청(Prediction Request) 보내기  

노래를 선택한 후, 우리는 추론을 위해 우리의 모델에 **Spotify ID**를 보내야 합니다. 

이를 하기 위해, 우리는 모델이 기대하는 올바른 **JSON 구조**로 입력 데이터를 포맷하는 함수를 정의합니다. 이 함수는 그러면 추론 엔드포인트로 **HTTP 요청**을 보내며, 모델은 요청을 처리하고 예측을 반환합니다. 

마지막으로, 우리는 **모델의 응답**을 추출하여 선택한 노래에 대한 예측된 결과를 얻습니다. 이 단계는 **Feast로 관리되는 기능**을 **실시간 머신러닝 예측**과 seamlessly 연결합니다.

In [ ]:
def rest_request(data):
    json_data = {
        "inputs": [
            {
                "name": "input",
                "shape": [1, 1],
                "datatype": "STRING",
                "data": data
            }
        ]
    }

    response = requests.post(infer_url, json=json_data, verify=False)
    response_dict = response.json()
    return response_dict['outputs'][0]['data']

## 📊 예측 얻기  

마지막으로, 우리는 우리의 선택한 노래의 **Spotify ID**를 모델로 보내고 **예측**을 얻습니다. 이 단계에서 모델은 노래의 기능을 처리하고 입력 데이터를 어떻게 해석하는지를 반영하는 예측을 반환합니다. 

우리가 받는 결과는 추천 생성이나 노래 순위 매기기 같은 작업에 사용될 수 있습니다. 이 마지막 단계로, 우리는 **Feast 기반의 Feature Store**를 **머신러닝 모델**과 성공적으로 연결했으며, 실시간 예측을 활성화했습니다! 🎶🚀


In [ ]:
data = favorite_song["spotify_id"].values
prediction = rest_request(data)
prediction